# 02 Remaining full grid on Kaggle: Qwen2.5-7B (rest), Llama-3.1-8B, Mistral-7B

**Why Kaggle.** The free Colab GPU quota ran out on 2026-10-12 with Qwen at about 1,422 of 1,632 runs. The remaining runs were completed on Kaggle's free T4, with the same setup as the Colab runs.

**Same setup as Colab.** The notebook stops unless all of these match:
- GPU: Tesla T4, one GPU (`CUDA_VISIBLE_DEVICES=0`), as on Colab.
- Ollama server 0.40.3 (pinned; the version recorded in the Colab pilot runs).
- Model digests from `configs/models.yaml`.
- Code: `code_hash` 332beea4f45e, the same as the pilot-v3 and Colab grid runs.

**Runs, in this order** (`configs/grid.yaml`; all 3 gates run by forking):
1. Qwen2.5-7B: the rest of its 1,632 runs (24 tasks; T=0 once and T=0.7 with 3 seeds). Finished Colab runs are reused.
2. Llama-3.1-8B: 204 runs (12 tasks, T=0). The 68 pilot-v3 runs are reused, so 136 are new.
3. Mistral-7B: 204 runs, the same as Llama.

**Expected time:** about 2 h plus about 10 min of setup (pilot v3: about 480 runs/h for Qwen and 300 runs/h for Llama).

**How to run**
1. Locally: download `MyDrive/p2-netops/results/traces/qwen25_7b/traces.jsonl` from Drive into `results/traces/qwen25_7b/`, then run `PYTHONPATH=src python scripts/make_kaggle_bundle.py`.
2. On kaggle.com, go to Datasets, then New dataset. Upload `dist/p2-netops-kaggle.zip` and keep it private.
3. Go to Code, then New notebook, then File, then Import notebook, and pick this file. Use Add input to attach the dataset.
4. In the notebook settings, set Accelerator to **GPU T4 x2** (not P100) and turn Internet **on**. Internet needs a phone-verified account.
5. Click Save Version, then Save & Run All (Commit). It runs in the background, so you can close the browser.

**Resumable.** If the run stops, add this notebook's output as a second input and run it again. Finished runs found in any input are skipped.

In [1]:
# Setup 1: find the bundle, copy the code, install Python deps
import os, shutil, subprocess, sys
from pathlib import Path
INPUT, WORK, P2 = Path('/kaggle/input'), Path('/kaggle/working'), Path('/tmp/p2')
shutil.rmtree(P2, ignore_errors=True)
found = sorted(INPUT.rglob('src/netinject/run.py'))
if found:  # Kaggle unpacked the uploaded zip
    shutil.copytree(found[0].parents[2], P2)
else:
    zips = sorted(INPUT.rglob('p2-netops-kaggle*.zip'))
    assert zips, 'Add the dataset holding p2-netops-kaggle.zip as notebook input first'
    shutil.unpack_archive(zips[0], P2)
!pip install -q -r {P2}/requirements-colab.txt
sys.path.insert(0, str(P2 / 'src'))
from netinject.run import code_hash
assert code_hash().startswith('332beea4f45e'), f'code_hash {code_hash()[:12]} differs from the Colab and pilot runs'

In [2]:
# Setup 2: install Ollama 0.40.3 (the server version of the Colab runs) and start it on one T4
!apt-get -qq install -y pciutils zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | OLLAMA_VERSION=0.40.3 sh
import time
from netinject.run import load_configs
GRID, MODELS = load_configs()   # all parameters come from configs/*.yaml
env = {**os.environ, 'CUDA_VISIBLE_DEVICES': '0', 'OLLAMA_NUM_PARALLEL': str(GRID['parallel']),
       'OLLAMA_MAX_LOADED_MODELS': '1', 'OLLAMA_KEEP_ALIVE': '-1'}
server = subprocess.Popen(['ollama', 'serve'], env=env,
                          stdout=open('/tmp/ollama.log', 'w'), stderr=subprocess.STDOUT)
time.sleep(8)

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [3]:
# Environment record (goes into the manifests)
import platform, importlib.metadata as md
GPU = subprocess.run(['nvidia-smi', '--id=0', '--query-gpu=name', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip()
OLLAMA_VERSION = subprocess.run(['ollama', '--version'], capture_output=True, text=True).stdout.strip()
print('GPU:', GPU)
print('Ollama server:', OLLAMA_VERSION, '| ollama-python', md.version('ollama'),
      '| pydantic', md.version('pydantic'), '| python', platform.python_version())
assert 'T4' in GPU, f'GPU is {GPU!r}: set Settings -> Accelerator -> GPU T4 x2'
assert OLLAMA_VERSION.endswith('0.40.3'), f'{OLLAMA_VERSION!r}: expected 0.40.3, as in the Colab runs'
print('Grid:', {k: GRID.get(k) for k in ('tasks', 'tasks_overrides', 'decoding', 'decoding_overrides', 'max_steps', 'max_calls_per_turn', 'num_ctx', 'parallel')})

GPU: Tesla T4
Ollama server: ollama version is 0.40.3 | ollama-python 0.6.3 | pydantic 2.13.5 | python 3.13.15
Grid: {'tasks': 'all', 'tasks_overrides': {'llama31_8b': ['T01', 'T03', 'T04', 'T05', 'T07', 'T11', 'T13', 'T15', 'T17', 'T19', 'T21', 'T22'], 'mistral_7b': ['T01', 'T03', 'T04', 'T05', 'T07', 'T11', 'T13', 'T15', 'T17', 'T19', 'T21', 'T22']}, 'decoding': [{'temperature': 0.0, 'runs': 1}, {'temperature': 0.7, 'runs': 3}], 'decoding_overrides': {'llama31_8b': [{'temperature': 0.0, 'runs': 1}], 'mistral_7b': [{'temperature': 0.0, 'runs': 1}]}, 'max_steps': 12, 'max_calls_per_turn': 1, 'num_ctx': 8192, 'parallel': 6}


In [4]:
# Run one model: copy finished runs found in the inputs, run the rest, write the manifest
import datetime, json, ollama
from collections import Counter
from netinject.evaluation.judge import read_records
from netinject.llm.ollama_client import OllamaClient
from netinject.run import finished_keys, plan, run_grid, write_manifest


def run_model(model_key):
    tag = MODELS[model_key]['tag']
    decoding = GRID.get('decoding_overrides', {}).get(model_key, GRID['decoding'])
    tasks = GRID.get('tasks_overrides', {}).get(model_key, GRID['tasks'])
    jobs = plan(None if tasks == 'all' else tasks, decoding)
    keys = {j[0] for j in jobs}
    print(model_key, 'decoding:', decoding, '|', len(jobs), 'runs')
    out = WORK / 'results' / 'traces' / model_key
    out.mkdir(parents=True, exist_ok=True)
    path = out / 'traces.jsonl'
    # Finished runs: prior/<model>/ in the bundle (Colab, pilot v3, tagged 'reused_from') and results/traces/<model>/
    # in an earlier output of this notebook. Only whole, error-free lines are copied, one per key.
    done = finished_keys(path)
    with open(path, 'a', encoding='utf-8') as f:
        for src in sorted(p for root in (P2, INPUT) for p in root.rglob(f'{model_key}/traces.jsonl')):
            for r in read_records(src):
                if r.get('error') is None and r['key'] in keys and r['key'] not in done:
                    f.write(json.dumps(r) + '\n')
                    done.add(r['key'])
    subprocess.run(['ollama', 'pull', tag], check=True, stdout=subprocess.DEVNULL)
    digest = {m.model: m.digest for m in ollama.Client().list().models}[tag]
    assert digest.startswith(MODELS[model_key]['digest_prefix']), f'{tag}: digest {digest} differs from configs/models.yaml'
    started = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds')
    run_grid(OllamaClient(tag, GRID['num_ctx'], GRID['num_predict']), model_key, path, jobs,
             GRID['max_steps'], GRID['parallel'], GRID['max_calls_per_turn'])
    finished = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds')
    m = write_manifest(out, f'02_{model_key}_kaggle_{started[:10]}', model_key, tag, digest, OLLAMA_VERSION, GPU,
                       jobs, started, finished)
    ok = {r['key']: r for r in read_records(path) if r.get('error') is None and r['key'] in keys}
    m.update(platform='kaggle', ollama_python=md.version('ollama'),
             runs_by_origin=dict(Counter(r.get('reused_from', 'kaggle') for r in ok.values())))
    (out / 'manifest.json').write_text(json.dumps(m, indent=2), encoding='utf-8')
    print(json.dumps({k: m[k] for k in ('episodes_expected', 'episodes_written', 'episodes_errored', 'runs_by_origin')}))


run_model('qwen25_7b')

qwen25_7b decoding: [{'temperature': 0.0, 'runs': 1}, {'temperature': 0.7, 'runs': 3}] | 1632 runs


pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest 
pulling 2bada8a74506:   2% ▕                  ▏  73 MB/4.7 GB                  pulling manifest 
pulling 2bada8a74506:   4% ▕                  ▏ 166 MB/4.7 GB                  pulling manifest 
pulling 2bada8a74506:   5% ▕                  ▏ 213 MB/4.7 GB                  pulling manifest 
pulling 2bada8a74506:   7% ▕█                 ▏ 307 MB/4.7 GB                  pulling manifest 
pulling 2bada8a74506:   9% ▕█                 ▏ 406 MB/4.7 GB                  pulling manifest 
pulling 2bada8a74506:  10% ▕█                 ▏ 460 MB/4.7 GB                  pulling manifest 
pulling 2bada8a74506:  12% ▕██                ▏ 570 MB/4.7 GB                  pulling manifest 
pulling 2bada8a74506:  14% ▕██                ▏ 649 MB/4.7 GB                  pulling manifest 
pulling 2bada8a74506:  15% ▕██                ▏ 700 MB/4.7 GB            

1632 jobs, 1422 already done, 210 to run
10/210 done, 225 runs/h, ETA 53 min
20/210 done, 292 runs/h, ETA 39 min
30/210 done, 362 runs/h, ETA 30 min
40/210 done, 411 runs/h, ETA 25 min
50/210 done, 452 runs/h, ETA 21 min
60/210 done, 477 runs/h, ETA 19 min
70/210 done, 504 runs/h, ETA 17 min
80/210 done, 507 runs/h, ETA 15 min
90/210 done, 489 runs/h, ETA 15 min
100/210 done, 504 runs/h, ETA 13 min
110/210 done, 512 runs/h, ETA 12 min
120/210 done, 505 runs/h, ETA 11 min
130/210 done, 508 runs/h, ETA 9 min
140/210 done, 508 runs/h, ETA 8 min
150/210 done, 507 runs/h, ETA 7 min
160/210 done, 514 runs/h, ETA 6 min
170/210 done, 517 runs/h, ETA 5 min
180/210 done, 518 runs/h, ETA 3 min
190/210 done, 520 runs/h, ETA 2 min
200/210 done, 525 runs/h, ETA 1 min
210/210 done, 528 runs/h, ETA 0 min
{"episodes_expected": 1632, "episodes_written": 1632, "episodes_errored": 0, "runs_by_origin": {"colab": 1422, "kaggle": 210}}


In [5]:
# 2 of 3: Llama-3.1-8B (68 pilot-v3 runs reused, 136 new)
run_model('llama31_8b')

llama31_8b decoding: [{'temperature': 0.0, 'runs': 1}] | 204 runs


pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest 
pulling 667b0c1932bc:   1% ▕                  ▏  72 MB/4.9 GB                  pulling manifest 
pulling 667b0c1932bc:   2% ▕                  ▏ 119 MB/4.9 GB                  pulling manifest 
pulling 667b0c1932bc:   5% ▕                  ▏ 226 MB/4.9 GB                  pulling manifest 
pulling 667b0c1932bc:   7% ▕█                 ▏ 323 MB/4.9 GB                  pulling manifest 
pulling 667b0c1932bc:   8% ▕█                 ▏ 373 MB/4.9 GB                  pulling manifest 
pulling 667b0c1932bc:  10% ▕█                 ▏ 469 MB/4.9 GB                  pulling manifest 
pulling 667b0c1932bc:  12% ▕██                ▏ 587 MB/4.9 GB                  pulling manifest 
pulling 667b0c1932bc:  13% ▕██                ▏ 632 MB/4.9 GB                  pulling manifest 
pulling 667b0c1932bc:  15% ▕██                ▏ 744 MB/4.9 GB            

204 jobs, 68 already done, 136 to run
10/136 done, 206 runs/h, ETA 37 min
20/136 done, 245 runs/h, ETA 28 min
30/136 done, 277 runs/h, ETA 23 min
40/136 done, 315 runs/h, ETA 18 min
50/136 done, 347 runs/h, ETA 15 min
60/136 done, 360 runs/h, ETA 13 min
70/136 done, 358 runs/h, ETA 11 min
80/136 done, 363 runs/h, ETA 9 min
90/136 done, 360 runs/h, ETA 8 min
100/136 done, 367 runs/h, ETA 6 min
110/136 done, 369 runs/h, ETA 4 min
120/136 done, 368 runs/h, ETA 3 min
130/136 done, 369 runs/h, ETA 1 min
136/136 done, 368 runs/h, ETA 0 min
{"episodes_expected": 204, "episodes_written": 204, "episodes_errored": 0, "runs_by_origin": {"pilot_v3": 68, "kaggle": 136}}


In [6]:
# 3 of 3: Mistral-7B (68 pilot-v3 runs reused, 136 new)
run_model('mistral_7b')

mistral_7b decoding: [{'temperature': 0.0, 'runs': 1}] | 204 runs


pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest 
pulling f5074b1221da:   0% ▕                  ▏  14 MB/4.4 GB                  pulling manifest 
pulling f5074b1221da:   2% ▕                  ▏  90 MB/4.4 GB                  pulling manifest 
pulling f5074b1221da:   3% ▕                  ▏ 129 MB/4.4 GB                  pulling manifest 
pulling f5074b1221da:   5% ▕                  ▏ 234 MB/4.4 GB                  pulling manifest 
pulling f5074b1221da:   7% ▕█                 ▏ 284 MB/4.4 GB                  pulling manifest 
pulling f5074b1221da:   9% ▕█                 ▏ 381 MB/4.4 GB                  pulling manifest 
pulling f5074b1221da:  11% ▕█                 ▏ 472 MB/4.4 GB                  pulling manifest 
pulling f5074b1221da:  12% ▕██                ▏ 531 MB/4.4 GB                  pulling manifest 
pulling f5074b1221da:  13% ▕██                ▏ 560 MB/4.4 GB            

204 jobs, 68 already done, 136 to run
10/136 done, 160 runs/h, ETA 47 min
20/136 done, 155 runs/h, ETA 45 min
30/136 done, 154 runs/h, ETA 41 min
40/136 done, 148 runs/h, ETA 39 min
50/136 done, 152 runs/h, ETA 34 min
60/136 done, 166 runs/h, ETA 27 min
70/136 done, 168 runs/h, ETA 24 min
80/136 done, 164 runs/h, ETA 20 min
90/136 done, 164 runs/h, ETA 17 min
100/136 done, 165 runs/h, ETA 13 min
110/136 done, 168 runs/h, ETA 9 min
120/136 done, 167 runs/h, ETA 6 min
130/136 done, 167 runs/h, ETA 2 min
136/136 done, 165 runs/h, ETA 0 min
{"episodes_expected": 204, "episodes_written": 203, "episodes_errored": 1, "runs_by_origin": {"pilot_v3": 68, "kaggle": 135}}


In [7]:
# Completeness check + quick look (informational; official labels and metrics are recomputed locally)
import pandas as pd
from netinject.evaluation.judge import label_file
for key in ('qwen25_7b', 'llama31_8b', 'mistral_7b'):
    out = WORK / 'results' / 'traces' / key
    m = json.loads((out / 'manifest.json').read_text())
    print(key, '| expected', m['episodes_expected'], '| written', m['episodes_written'], '| errored', m['episodes_errored'],
          '|', 'COMPLETE' if m['episodes_written'] == m['episodes_expected'] else 'INCOMPLETE: run again to resume')
    df = label_file(out / 'traces.jsonl')
    att = df[df['style'] != 'control']
    print(att.pivot_table(index='gate', columns='delay', values='unsafe_write_executed', aggfunc='mean').round(3))

qwen25_7b | expected 1632 | written 1632 | errored 0 | COMPLETE
delay           0.0    1.0    2.0    3.0
gate                                    
none          0.245  0.177  0.133  0.099
scope_check   0.036  0.003  0.008  0.010
ticket_bound  0.000  0.000  0.000  0.000
llama31_8b | expected 204 | written 204 | errored 0 | COMPLETE
delay           0.0    1.0    2.0    3.0
gate                                    
none          0.250  0.104  0.125  0.083
scope_check   0.021  0.000  0.000  0.000
ticket_bound  0.000  0.000  0.000  0.000
mistral_7b | expected 204 | written 203 | errored 1 | INCOMPLETE: run again to resume
delay           0.0  1.0  2.0  3.0
gate                              
none          0.021  0.0  0.0  0.0
scope_check   0.021  0.0  0.0  0.0
ticket_bound  0.000  0.0  0.0  0.0


In [8]:
# One zip of all three models' traces for download (Output tab)
zip_path = shutil.make_archive(str(WORK / 'results_kaggle'), 'zip', root_dir=WORK / 'results', base_dir='traces')
print('Download', zip_path)

Download /kaggle/working/results_kaggle.zip


## After the run
1. Download the executed notebook from the version page and save it as `notebooks/kaggle/02_remaining_kaggle.ipynb`, replacing this copy.
2. Download `results_kaggle.zip` from the Output tab and unzip it into the repo's `results/` folder. That gives `results/traces/{qwen25_7b,llama31_8b,mistral_7b}/`, and the complete Qwen file replaces the partial one.